In [77]:
#importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import datetime
from datetime import datetime

In [78]:
#import data set
df = pd.read_csv('/content/sample_data/Invoice_Detail.csv', on_bad_lines='skip', engine='python')
df_Assignement = pd.read_excel('/content/sample_data/Assignement File.xlsx', engine='openpyxl')

In [79]:
# Drop column
df = df.drop(columns=['HCPCS'])

In [80]:
df = df[~df['Status'].isin(['Closed', 'Rejected'])]

In [81]:
# Convert blank/whitespace-only strings to NaN
df = df.replace(r'^\s*$', pd.NA, regex=True)

# Drop rows where ANY column has a null/blank value
df = df.dropna()

In [82]:
# Sort A to Z by Patient Name
df = df.sort_values(by="Patient Name")

In [83]:
# Remove rows where Balance is negative
df = df[df['Balance'] > 0]
df = df[df['Charge'] > 0]

In [84]:
# ---- Step 1: Calculate total charge by Invoice ----
summary = df.groupby("Invoice", as_index=False).agg(
    Total_Charge=("Charge", "sum")
)

# ---- Step 2: Merge total charge back into original file ----
df = df.merge(summary, on="Invoice", how="left")

In [85]:
# ---- Step 1: Calculate total balance by Invoice ----
summary = df.groupby("Invoice", as_index=False).agg(
    Total_Balance=("Balance", "sum")
)

# ---- Step 2: Merge total balance back into original file ----
df = df.merge(summary, on="Invoice", how="left")

In [86]:
# Remove duplicate invoices (keep first occurrence)
df = df.drop_duplicates(subset=["Invoice"], keep="first")

In [87]:
# Convert DOS From column to datetime
df["DOS From"] = pd.to_datetime(df["DOS From"], errors="coerce")

# Today's date as a pandas Timestamp
today_ts = pd.Timestamp(datetime.today().date())

# Calculate difference in days
df["Age of Claims"] = (today_ts - df["DOS From"]).dt.days

# Sort from 0 to infinity
df = df.sort_values(by="Age of Claims", ascending=True)

In [88]:
# Correct bucket logic
conditions = [
    (df["Age of Claims"] >= 0) & (df["Age of Claims"] <= 30),
    (df["Age of Claims"] > 30) & (df["Age of Claims"] <= 60),
    (df["Age of Claims"] > 60) & (df["Age of Claims"] <= 90),
    (df["Age of Claims"] > 90) & (df["Age of Claims"] <= 120),
    (df["Age of Claims"] > 120)
]

bucket_labels = [
    "Bucket - 1",
    "Bucket - 2",
    "Bucket - 3",
    "Bucket - 4",
    "Bucket - 5"
]

# Assign bucket column
df["Bucket"] = np.select(conditions, bucket_labels, default="Uncategorized")

In [89]:
# 7. Merge Assignment File & Add Rep Column
df = df.merge(
    df_Assignement[["Payer Name", "Reps"]],
    on="Payer Name",
    how="left"
)

In [90]:
df.head()

,Invoice,Patient Name,Payer Name,DOS From,Charge,Balance,Status,Total_Charge,Total_Balance,Age of Claims,Bucket,Reps
0,IN-9cef7357,YOUSSEF MAHGEREFTEH,DMERC - Region D,2025-10-01,27.54,27.54,Submitted,177.54,177.54,51,Bucket - 2,MedKarma Rep 2
1,IN-2252b4bc,Oscar Romo,LA Care,2025-10-01,95.85,95.85,Denied,95.85,95.85,51,Bucket - 2,MedKarma Rep 1
2,IN-264209dd,Albrae Reynolds,DMERC - Region D,2025-10-01,104.26,104.26,Submitted,104.26,104.26,51,Bucket - 2,MedKarma Rep 2
3,IN-f323f6b8,LILLIE MAE JONES BUNKLEY,DMERC - Region D,2025-10-01,150.00,150.00,Submitted,192.00,192.00,51,Bucket - 2,MedKarma Rep 2
4,IN-a73fc643,Carrie Martin,DMERC - Region D,2025-10-01,116.03,116.03,Submitted,116.03,116.03,51,Bucket - 2,MedKarma Rep 2


In [91]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3420 entries, 0 to 3419
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Invoice        3420 non-null   object        
 1   Patient Name   3420 non-null   object        
 2   Payer Name     3420 non-null   object        
 3   DOS From       3420 non-null   datetime64[ns]
 4   Charge         3420 non-null   float64       
 5   Balance        3420 non-null   float64       
 6   Status         3420 non-null   object        
 7   Total_Charge   3420 non-null   float64       
 8   Total_Balance  3420 non-null   float64       
 9   Age of Claims  3420 non-null   int64         
 10  Bucket         3420 non-null   object        
 11  Reps           3327 non-null   object        
dtypes: datetime64[ns](1), float64(4), int64(1), object(6)
memory usage: 320.8+ KB


In [92]:
output_path = '/content/Invoice_Clean.csv'
df.to_csv(output_path, index=False)
print(f"\n✅ Cleaned data saved successfully to: {output_path}")


✅ Cleaned data saved successfully to: /content/Invoice_Clean.csv
